# Tier 0 Hawkes Baseline -- Cascade Graph Walkthrough

Loads the most recent Tier 0 checkpoint from `runs/tier0/` and inspects the learned alpha matrix as a cascade graph.

**Note:** this notebook expects `runs/tier0/<timestamp>/params.pkl` to exist. Run `eonet model train-hawkes` first.


In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

from eonet_cascades.interpret.excitation import excitation_to_dataframe, plot_excitation_heatmap

runs = sorted(Path('runs/tier0').glob('*/params.pkl'))
if not runs:
    raise FileNotFoundError('No Tier 0 checkpoint found. Run `eonet model train-hawkes` first.')
latest = runs[-1]
with open(latest, 'rb') as f:
    ckpt = pickle.load(f)
print('Loaded:', latest)
print('Window:', ckpt['window'])
print('Marks:', ckpt['mark_names'])
print('N events used:', ckpt['n_events_used'])
print('Final NLL:', ckpt['fit_result']['nll_final'])
print('Spectral radius:', ckpt['fit_result'].get('spectral_radius'))
print('Status:', ckpt['fit_result']['status'])


In [ ]:
fig = plot_excitation_heatmap(ckpt['params'], ckpt['mark_names'])
plt.show()


In [ ]:
df = excitation_to_dataframe(ckpt['params'], ckpt['mark_names'])
print('Top 15 parent -> child triggers by alpha:')
print(df.sort('alpha', descending=True).head(15))


## Cascades to inspect

For each non-trivial (parent -> child) pair pulled from the table above, evaluate qualitatively:

- Is the trigger plausible physically? (e.g. wildfire -> wildfire = within-fire-complex propagation, OK. Earthquake -> wildfire = bogus, model artifact.)
- Does the temporal kernel (1/beta = mean delay in days) make sense for that pair?
- Does the spatial bandwidth sigma match the scale of the underlying process?

Write a 2-sentence note per pair below as a record of what we learned from this fit.
